# Custom attention in ~30 lines — HeliosLM

CPU-first notebook for [Issue #3](https://github.com/tonythetiger168/helioslm/issues/3). Run from the repository root with `jupyter notebook examples/notebooks/custom_attention.ipynb`. The variant wraps MLA with a trainable per-channel output gain initialized to the identity; it is a drop-in at initialization, not a new attention algorithm or a performance claim. It preserves the MLA cache contract.

In [ ]:
import json, sys, time, statistics
from pathlib import Path
import torch
from torch import nn
root = Path.cwd()
if not (root / 'helioslm_v5').is_dir():
    root = root.parent.parent
sys.path.insert(0, str(root))
from helioslm_v5.configs.config_v5 import HeliosLMv5Config
from helioslm_v5.src.attention.mla import MLA
cfg = HeliosLMv5Config(size='lite')
torch.manual_seed(7)
print('torch', torch.__version__, 'hidden', cfg.hidden_size)

## Minimal drop-in variant

A learnable residual gain on the MLA output. At zero initialization it is exactly the same function as MLA, including cached decoding. For tuple returns, only the attention output is transformed; caches are returned unchanged.

In [ ]:
class GainMLA(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = MLA(config)
        self.gain = nn.Parameter(torch.zeros(config.hidden_size))

    def forward(self, hidden_states, attention_mask=None, past_key_value=None,
                use_cache=False, position_ids=None, shared_kv=None):
        result = self.attn(hidden_states, attention_mask=attention_mask,
                           past_key_value=past_key_value, use_cache=use_cache,
                           position_ids=position_ids, shared_kv=shared_kv)
        if isinstance(result, tuple):
            output, *rest = result
            return (output * (1 + self.gain), *rest)
        return result * (1 + self.gain)

base = MLA(cfg).eval()
variant = GainMLA(cfg).eval()
variant.attn.load_state_dict(base.state_dict())

## Oracle-style equivalence and nontrivial perturbation

Follow the project's deterministic, tight-tolerance convention. Verify prefill and cache, then confirm the variant actually changes behavior after updating its gain.

In [ ]:
x = torch.randn(1, 8, cfg.hidden_size)
with torch.inference_mode():
    ref = base(x, use_cache=True)
    got = variant(x, use_cache=True)
assert isinstance(ref, tuple) and isinstance(got, tuple)
torch.testing.assert_close(got[0], ref[0], atol=1e-6, rtol=1e-6)
assert len(ref) == len(got)
for a, b in zip(ref[1:], got[1:]):
    if isinstance(a, (tuple, list)):
        for ai, bi in zip(a, b):
            torch.testing.assert_close(ai, bi, atol=0, rtol=0)
    elif isinstance(a, torch.Tensor):
        torch.testing.assert_close(a, b, atol=0, rtol=0)
with torch.no_grad():
    variant.gain.fill_(0.1)
with torch.inference_mode():
    changed = variant(x, use_cache=False)
    original = base(x, use_cache=False)
assert not torch.allclose(changed[0] if isinstance(changed, tuple) else changed,
                          original[0] if isinstance(original, tuple) else original)
variant.gain.data.zero_()
print('PASS: identity oracle, cache parity and gain perturbation')

## CPU latency comparison

These timings are measurements on your machine, not hard-coded results. The gain wrapper adds overhead and is **not** expected to beat MLA.

In [ ]:
torch.set_num_threads(min(torch.get_num_threads(), 4))
def bench(module, repeats=20, warmup=5):
    with torch.inference_mode():
        for _ in range(warmup): module(x, use_cache=False)
        durations = []
        for _ in range(repeats):
            t = time.perf_counter()
            module(x, use_cache=False)
            durations.append((time.perf_counter()-t)*1000)
    return statistics.median(durations)
result = {'torch': torch.__version__, 'device': 'cpu', 'seed': 7,
          'shape': list(x.shape), 'mla_median_ms': bench(base),
          'gain_mla_median_ms': bench(variant)}
result['ratio_variant_to_mla'] = result['gain_mla_median_ms']/result['mla_median_ms']
print(json.dumps(result, indent=2))
import os
# HELIOS_NOTEBOOK_OUT redirects the result file (the test suite points it
# at a temporary directory so parallel test runs never write benchmarks/)
out_dir = Path(os.environ.get('HELIOS_NOTEBOOK_OUT') or os.environ.get('EVASTAR_NOTEBOOK_OUT', root / 'benchmarks'))
out = out_dir / 'custom_attention_cpu_results.json'
out.write_text(json.dumps(result, indent=2)+'\n', encoding='utf-8')
print('Saved:', out)